# Прогнозирование добычи газа

Учебный проект на основе предоставленного набора данных. Результаты относятся к исследованному набору и выбранному разбиению; для переноса на новые данные нужна дополнительная проверка.

In [ ]:
from pathlib import Path
DATA_PATH = Path('data/well_data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Поместите исходный CSV в {DATA_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import sklearn
from importlib.metadata import version
for package in ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy', 'scikit-learn']:
    print(package, version(package))

In [ ]:
df = pd.read_csv(DATA_PATH)
assert 'extr_per_day' in df.columns, 'Отсутствует целевой столбец'
print('Данные загружены в df.')


## Проверка качества данных

In [ ]:
df.sample(10)

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df[df["permeability"] > 8.5]

In [ ]:
df[(df['acoustic_impedance'] < 1.35125) | (df['acoustic_impedance'] > 4.54125)]

In [ ]:
df[df["organic_carbon"] < 0]

In [ ]:
df_clean = df.copy()
df_clean.loc[df_clean['organic_carbon'] < 0, 'organic_carbon'] = np.nan
print('Пропусков organic_carbon:', df_clean['organic_carbon'].isna().sum())

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

**Вывод:** В исходной таблице 200 строк и 8 столбцов; пропусков до замены отрицательных значений и полных дубликатов строк не обнаружено.

В organic_carbon обнаружены четыре физически невозможных отрицательных значения: −0,04, −0,19, −0,03 и −0,04. В копии они отмечены как пропуски и после разделения заполнены медианой допустимых значений train; та же медиана применяется к test.

Проверены пять наблюдений с permeability выше 8,5 и два значения acoustic_impedance за границами 1,5 IQR. Они оставлены: необычное значение само по себе не доказывает ошибку, а подтверждения ошибки измерений нет. Для brittleness и extr_per_day минимум и максимум внутри границ 1,5 IQR; их также оставили. Эти проверки не доказывают достоверность каждого измерения.

well_id сохраняется для идентификации скважин, но не включается в X. Целевая переменная extr_per_day также исключается из X.

## Исследовательский анализ данных

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df_clean, x='extr_per_day', bins=20)
plt.title('Распределение суточной добычи')
plt.xlabel('Добыча в сутки (extr_per_day)')
plt.ylabel('Число скважин')
plt.tight_layout()
plt.show()

In [ ]:
features = ['porosity', 'permeability', 'acoustic_impedance', 'brittleness', 'organic_carbon', 'vitrinite_reflectance']
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for ax, feature in zip(axes.flat, features):
    sns.histplot(df_clean[feature].dropna(), bins=20, ax=ax)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Число скважин')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for ax, feature in zip(axes.flat, features):
    sns.scatterplot(data=df_clean, x=feature, y='extr_per_day', ax=ax, alpha=0.7)
    ax.set_title(f'{feature} и добыча')
    ax.set_xlabel(feature)
    ax.set_ylabel('Добыча в сутки (extr_per_day)')
fig.tight_layout()
plt.show()

In [ ]:
rock_features = ['porosity', 'permeability', 'acoustic_impedance', 'brittleness', 'organic_carbon', 'vitrinite_reflectance']
sns.pairplot(df_clean[rock_features].dropna(), diag_kind='hist', plot_kws={'alpha': 0.6, 's': 18})
plt.show()

In [ ]:
corr = df_clean.drop(columns='well_id').corr(method='pearson')
display(corr)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Корреляция Пирсона')
plt.tight_layout()
plt.show()

**Вывод:** На гистограмме суточной добычи виден один основной пик; большинство значений находится приблизительно в диапазоне 3000–6000. Явных изолированных экстремальных значений визуально не видно.

На диаграммах рассеяния пористость показывает сильную положительную связь с добычей; проницаемость и органический углерод — положительную связь с разбросом. Для акустического импеданса видна отрицательная тенденция, для хрупкости явной линейной связи на глаз почти нет, а связь отражения витринита слабее и неоднозначна. Графики не доказывают причинность.

Корреляция Пирсона между porosity и permeability составляет около 0,76 и превышает условный порог 0,7: признаки часто растут вместе. Это указывает на возможную мультиколлинеарность, которая может затруднять устойчивую оценку отдельных коэффициентов модели. Корреляция не устанавливает причинность; решение о признаках рассмотрено при подготовке модели.

На парных графиках признаков заметна положительная связь porosity и permeability; porosity и organic_carbon также связаны положительно (r ≈ 0,69). Для acoustic_impedance с porosity и organic_carbon наблюдается отрицательная тенденция. На графиках присутствует разброс; причинность эти связи не устанавливают.

## Подготовка признаков

In [ ]:
X = df_clean.drop(columns=['well_id', 'extr_per_day']).copy()
y = df_clean['extr_per_day'].copy()
print('Размер X:', X.shape)
print('Признаки:', X.columns.tolist())
print('Пропусков organic_carbon:', X['organic_carbon'].isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
feature_sets = {
    'Все признаки': X_train.columns.tolist(),
    'Без permeability': X_train.drop(columns='permeability').columns.tolist(),
    'Без porosity': X_train.drop(columns='porosity').columns.tolist()
}
results = []
for name, columns in feature_sets.items():
    pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LinearRegression())
    scores = cross_validate(
        pipeline, X_train[columns], y_train, cv=cv,
        scoring={'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}
    )
    rmse = -scores['test_rmse']
    results.append({
        'Набор': name, 'RMSE среднее': rmse.mean(), 'RMSE std': rmse.std(),
        'R² среднее': scores['test_r2'].mean(), 'R² std': scores['test_r2'].std()
    })
print(pd.DataFrame(results).set_index('Набор').round(3).to_string())

In [ ]:
imputer_vif = SimpleImputer(strategy='median')
X_vif = pd.DataFrame(imputer_vif.fit_transform(X_train),
                     columns=X_train.columns, index=X_train.index)
vif_rows = []
for feature in X_vif.columns:
    others = X_vif.drop(columns=feature)
    r2 = LinearRegression(fit_intercept=True).fit(others, X_vif[feature]).score(others, X_vif[feature])
    vif = np.inf if 1 - r2 <= 1e-12 else 1 / (1 - r2)
    vif_rows.append({'Признак': feature, 'VIF': vif})
print(pd.DataFrame(vif_rows).set_index('Признак').round(3).to_string())

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_train.columns, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_train.columns, index=X_test.index)

X_train_with_intercept = X_train_scaled.copy()
X_test_with_intercept = X_test_scaled.copy()
X_train_with_intercept.insert(0, 'intercept', 1.0)
X_test_with_intercept.insert(0, 'intercept', 1.0)

assert X_train_scaled.columns.equals(X_test_scaled.columns)
assert not X_train_scaled.isna().any().any() and not X_test_scaled.isna().any().any()
assert (X_train_with_intercept['intercept'] == 1).all() and (X_test_with_intercept['intercept'] == 1).all()
assert X_train_with_intercept.shape == (len(y_train), X.shape[1] + 1)
assert X_test_with_intercept.shape == (len(y_test), X.shape[1] + 1)
print('Train и test:', X_train_scaled.shape, X_test_scaled.shape)
print('Для ручных методов:', X_train_with_intercept.shape, X_test_with_intercept.shape)
print('Пропусков после обработки:', X_train_scaled.isna().sum().sum(), X_test_scaled.isna().sum().sum())
print('Медиана organic_carbon по train:', imputer.statistics_[X_train.columns.get_loc('organic_carbon')])
print('Столбцы:', X_train_with_intercept.columns.tolist())

In [ ]:
X_train_reduced = X_train.drop(columns='permeability').copy()
X_test_reduced = X_test.drop(columns='permeability').copy()
imputer_reduced = SimpleImputer(strategy='median')
train_reduced_imputed = pd.DataFrame(imputer_reduced.fit_transform(X_train_reduced), columns=X_train_reduced.columns, index=X_train_reduced.index)
test_reduced_imputed = pd.DataFrame(imputer_reduced.transform(X_test_reduced), columns=X_train_reduced.columns, index=X_test_reduced.index)
scaler_reduced = StandardScaler()
X_train_reduced_scaled = pd.DataFrame(scaler_reduced.fit_transform(train_reduced_imputed), columns=X_train_reduced.columns, index=X_train_reduced.index)
X_test_reduced_scaled = pd.DataFrame(scaler_reduced.transform(test_reduced_imputed), columns=X_train_reduced.columns, index=X_test_reduced.index)
X_train_reduced_intercept = X_train_reduced_scaled.copy()
X_test_reduced_intercept = X_test_reduced_scaled.copy()
X_train_reduced_intercept.insert(0, 'intercept', 1.0)
X_test_reduced_intercept.insert(0, 'intercept', 1.0)
assert X_train_reduced_intercept.shape == (160, 6) and X_test_reduced_intercept.shape == (40, 6)
assert X_train_reduced_scaled.columns.equals(X_test_reduced_scaled.columns)
assert not X_train_reduced_scaled.isna().any().any() and not X_test_reduced_scaled.isna().any().any()
assert (X_train_reduced_intercept['intercept'] == 1).all() and (X_test_reduced_intercept['intercept'] == 1).all()
corr_reduced = X_train_reduced.corr()
upper = corr_reduced.where(np.triu(np.ones(corr_reduced.shape), k=1).astype(bool))
vif_reduced = {}
for feature in train_reduced_imputed.columns:
    others = train_reduced_imputed.drop(columns=feature)
    r2 = LinearRegression().fit(others, train_reduced_imputed[feature]).score(others, train_reduced_imputed[feature])
    vif_reduced[feature] = np.inf if 1 - r2 <= 1e-12 else 1 / (1 - r2)
print('Формы train/test:', X_train_reduced_intercept.shape, X_test_reduced_intercept.shape)
print('Корреляции train (попарно без пропусков):')
print(corr_reduced.round(3).to_string())
print('Максимум |r| вне диагонали:', upper.abs().max().max())
print('VIF после заполнения медианой train:')
print(pd.Series(vif_reduced).round(3).to_string())

**Вывод:** Из очищенной копии сформированы X и y; well_id и целевой extr_per_day исключены из признаков. Данные разделены на train и test в соотношении 80/20 с random_state=42.

В полном наборе корреляция porosity и permeability около 0,76 превышает условный порог 0,7. Для сравнения создана параллельная ветка без permeability. Она выбрана из двух вариантов по пятифолдовой CV на train: RMSE без permeability 238,4, без porosity 405,2; полный набор дал 205,2. Значит, удаление ухудшило прогноз в этом эксперименте, но разорвало выявленную пару. На train максимальная оставшаяся попарная корреляция по модулю 0,675, максимальный VIF после заполнения — 3,98. Это не доказывает полного отсутствия многомерной коллинеарности. Полный набор сохранён для сравнения в анализе регуляризации.

Для каждой ветки медианы и StandardScaler обучены только на train и применены к test; для organic_carbon медиана train — 1,05. Для ручных методов добавлен единичный столбец. Исходный EDA проводился до разделения по всей таблице, поэтому test уже был просмотрен и не является полностью независимым для окончательного выбора; CV и подбор параметров используют только train.

## Линейная регрессия методом наименьших квадратов

*Базовая формула:*

$w=(X^TX)^{−1}X^Ty$

Здесь X содержит единичный столбец. Для вычисления используется np.linalg.lstsq: при полном ранге матрицы X решение эквивалентно формуле выше, но не требует явного обращения XᵀX и устойчивее численно.

In [ ]:
coef_ols, residuals, rank, singular_values = np.linalg.lstsq(X_train_with_intercept.to_numpy(), y_train.to_numpy(), rcond=None)
y_pred_ols_train = X_train_with_intercept.to_numpy() @ coef_ols
y_pred_ols_test = X_test_with_intercept.to_numpy() @ coef_ols

model_sklearn = LinearRegression(fit_intercept=True)
model_sklearn.fit(X_train_scaled, y_train)
y_train_sklearn = model_sklearn.predict(X_train_scaled)
y_test_sklearn = model_sklearn.predict(X_test_scaled)
w_sklearn = np.r_[model_sklearn.intercept_, model_sklearn.coef_]

comparison = pd.DataFrame({'МНК': coef_ols, 'sklearn': w_sklearn, 'Разница': coef_ols - w_sklearn}, index=X_train_with_intercept.columns)
print('Ранг матрицы:', rank, 'из', X_train_with_intercept.shape[1])
print(comparison.to_string())
print('Максимальная разница коэффициентов:', np.max(np.abs(coef_ols - w_sklearn)))
print('Максимальная разница прогнозов train:', np.max(np.abs(y_pred_ols_train - y_train_sklearn)))
print('Максимальная разница прогнозов test:', np.max(np.abs(y_pred_ols_test - y_test_sklearn)))
assert np.allclose(coef_ols, w_sklearn, rtol=1e-8, atol=1e-8)
assert np.allclose(y_pred_ols_train, y_train_sklearn, rtol=1e-8, atol=1e-8)
assert np.allclose(y_pred_ols_test, y_test_sklearn, rtol=1e-8, atol=1e-8)

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

metrics_ols = pd.DataFrame({
    'RMSE': [root_mean_squared_error(y_train, y_pred_ols_train), root_mean_squared_error(y_test, y_pred_ols_test)],
    'R²': [r2_score(y_train, y_pred_ols_train), r2_score(y_test, y_pred_ols_test)]
}, index=['Train', 'Test'])
print(metrics_ols.to_string())

In [ ]:
coef_analysis = pd.DataFrame({
    'Коэффициент (стандартизированный признак)': coef_ols[1:],
    'Корреляция Пирсона с добычей (train)': X_train.corrwith(y_train),
    'Число пар без пропусков': X_train.notna().mul(y_train.notna(), axis=0).sum()
}, index=X_train_scaled.columns)
coef_analysis['Модуль коэффициента'] = coef_analysis['Коэффициент (стандартизированный признак)'].abs()
coef_analysis = coef_analysis.sort_values('Модуль коэффициента', ascending=False)
print('Корреляции рассчитаны на train попарно по непустым значениям; organic_carbon — без пропусков.')
print(coef_analysis.round(4).to_string())

In [ ]:
A_train_reduced = X_train_reduced_intercept.to_numpy()
A_test_reduced = X_test_reduced_intercept.to_numpy()
coef_reduced, _, rank_reduced, _ = np.linalg.lstsq(A_train_reduced, y_train.to_numpy(), rcond=None)
model_reduced = LinearRegression().fit(X_train_reduced_scaled, y_train)
coef_reduced_sklearn = np.r_[model_reduced.intercept_, model_reduced.coef_]
pred_reduced_train = A_train_reduced @ coef_reduced
pred_reduced_test = A_test_reduced @ coef_reduced
assert rank_reduced == A_train_reduced.shape[1]
assert np.allclose(coef_reduced, coef_reduced_sklearn, atol=1e-8, rtol=1e-8)
assert np.allclose(pred_reduced_test, model_reduced.predict(X_test_reduced_scaled))
print('Ранг:', rank_reduced, 'из', A_train_reduced.shape[1])
print('Максимальная разница коэффициентов МНК и sklearn:', np.max(np.abs(coef_reduced - coef_reduced_sklearn)))
print(pd.DataFrame({'RMSE': [root_mean_squared_error(y_train, pred_reduced_train), root_mean_squared_error(y_test, pred_reduced_test)], 'R²': [r2_score(y_train, pred_reduced_train), r2_score(y_test, pred_reduced_test)]}, index=['Train', 'Test']).to_string())
coef_reduced_table = pd.DataFrame({'Коэффициент': coef_reduced[1:], 'Корреляция с добычей train': X_train_reduced.corrwith(y_train)}, index=X_train_reduced.columns)
print(coef_reduced_table.round(4).to_string())

**Вывод:** Выбраны RMSE и R². RMSE выражается в единицах целевой переменной и сильнее учитывает крупные ошибки; R² сравнивает сумму квадратов ошибок с прогнозом средним целевой переменной соответствующей выборки.

Для полного набора RMSE/R² на train — 197,66/0,9613, на test — 208,98/0,9484. Для сокращённого набора МНК и sklearn совпадают с точностью около 1,8·10⁻¹², ранг 6 из 6; RMSE/R² на train — 230,36/0,9475, на test — 237,50/0,9333. На этом разбиении сокращение ухудшило прогноз. Один test из 40 наблюдений не гарантирует такого же качества на новых данных.

В сокращённой модели наибольший стандартизированный коэффициент у porosity (858,08); её корреляция с добычей на train — 0,874. У organic_carbon корреляция 0,637 при коэффициенте 76,35. В полной модели коэффициент porosity равен 685,29, а permeability — 194,72. Коэффициенты отражают условную связь при фиксированных остальных признаках и могут изменяться при изменении набора признаков; они не доказывают причинность. Исходный EDA включал test, поэтому оценка не полностью независима.

## Градиентный спуск

In [ ]:
def batch_gradient_descent(X, y, learning_rate, iterations):
    weights = np.zeros(X.shape[1])
    history = []
    weights_history = []
    for step in range(iterations + 1):
        errors = X @ weights - y
        mse = np.mean(errors ** 2)
        if not np.isfinite(mse) or mse > 1e100:
            last_weights = weights_history[-1] if weights_history else weights
            return last_weights, np.array(history), np.array(weights_history).reshape(-1, X.shape[1]), 'расхождение'
        history.append(mse)
        weights_history.append(weights.copy())
        if step < iterations:
            gradient = (2 / len(y)) * X.T @ errors
            weights = weights - learning_rate * gradient
    return weights, np.array(history), np.array(weights_history), 'завершено'

X_gd_train = X_train_with_intercept.to_numpy()
y_gd_train = y_train.to_numpy()
mse_ols_train = np.mean((y_gd_train - y_pred_ols_train) ** 2)
learning_rates = [0.001, 0.01, 0.1, 0.3]
iterations = 3000
gd_results = {}
rows = []
for rate in learning_rates:
    weights, history, weight_history, status = batch_gradient_descent(X_gd_train, y_gd_train, rate, iterations)
    gd_results[rate] = {'weights': weights, 'mse': history, 'weight_history': weight_history, 'status': status}
    rows.append({'Шаг': rate, 'Итераций': len(history) - 1, 'Статус': status,
                 'MSE train': history[-1] if len(history) else np.nan,
                 'Разница с МНК': history[-1] - mse_ols_train if len(history) else np.nan,
                 'Макс. разница коэффициентов': np.max(np.abs(weights - coef_ols)) if status == 'завершено' else np.nan})

print(f'MSE МНК на train: {mse_ols_train:.6f}')
print(pd.DataFrame(rows).to_string(index=False))
plt.figure(figsize=(9, 5))
for rate, result in gd_results.items():
    plt.plot(result['mse'], label=f'Шаг {rate}')
plt.axhline(mse_ols_train, color='black', linestyle='--', label='МНК train')
plt.yscale('log')
plt.xlabel('Итерация')
plt.ylabel('MSE на train (логарифмическая шкала)')
plt.title('Сходимость batch градиентного спуска')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
learning_rate_selected = 0.1
iterations_selected = 3000
X_gd_test = X_test_with_intercept.to_numpy()
y_gd_test = y_test.to_numpy()
selected_result = gd_results[learning_rate_selected]
assert selected_result['status'] == 'завершено'
weights_gd = selected_result['weights']
weights_history = selected_result['weight_history']
assert len(weights_history) == iterations_selected + 1
assert np.all(weights_history[0] == 0)
assert np.array_equal(weights_history[-1], weights_gd)
mse_train_history = np.mean((X_gd_train @ weights_history.T - y_gd_train[:, None]) ** 2, axis=0)
mse_test_history = np.mean((X_gd_test @ weights_history.T - y_gd_test[:, None]) ** 2, axis=0)
assert len(mse_train_history) == iterations_selected + 1
assert np.allclose(mse_train_history, gd_results[learning_rate_selected]['mse'])
y_pred_gd_train = X_gd_train @ weights_gd
y_pred_gd_test = X_gd_test @ weights_gd
metrics_comparison = pd.DataFrame([
    ['GD', 'Train', root_mean_squared_error(y_train, y_pred_gd_train), r2_score(y_train, y_pred_gd_train)],
    ['МНК', 'Train', root_mean_squared_error(y_train, y_pred_ols_train), r2_score(y_train, y_pred_ols_train)],
    ['GD', 'Test', root_mean_squared_error(y_test, y_pred_gd_test), r2_score(y_test, y_pred_gd_test)],
    ['МНК', 'Test', root_mean_squared_error(y_test, y_pred_ols_test), r2_score(y_test, y_pred_ols_test)]
], columns=['Модель', 'Выборка', 'RMSE', 'R²'])
print('Шаг 0.1 и 3000 итераций выбраны по train; test показан после выбора.')
print(metrics_comparison.to_string(index=False))
print('Максимальная разница коэффициентов GD и МНК:', np.max(np.abs(weights_gd - coef_ols)))
plt.figure(figsize=(9, 5))
plt.plot(mse_train_history, label='Train')
plt.plot(mse_test_history, label='Test')
plt.yscale('log')
plt.xlabel('Итерация')
plt.ylabel('MSE (логарифмическая шкала)')
plt.title('Сходимость batch GD: шаг 0.1, 3000 итераций')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gd_reduced_results = {}
for rate in [0.001, 0.01, 0.1, 0.3]:
    w, history, wh, status = batch_gradient_descent(A_train_reduced, y_train.to_numpy(), rate, 3000)
    gd_reduced_results[rate] = (w, history, wh, status)
    print('Шаг:', rate, 'статус:', status, 'разница коэффициентов с МНК:', np.max(np.abs(w - coef_reduced)))
rate_reduced = min([r for r in gd_reduced_results if gd_reduced_results[r][3] == 'завершено'], key=lambda r: np.max(np.abs(gd_reduced_results[r][0] - coef_reduced)))
w_gd_reduced, history_reduced, wh_reduced, _ = gd_reduced_results[rate_reduced]
pred_gd_reduced_train = A_train_reduced @ w_gd_reduced
pred_gd_reduced_test = A_test_reduced @ w_gd_reduced
print('Шаг выбран по совпадению с МНК на train:', rate_reduced)
print(pd.DataFrame({'RMSE': [root_mean_squared_error(y_train, pred_gd_reduced_train), root_mean_squared_error(y_test, pred_gd_reduced_test)], 'R²': [r2_score(y_train, pred_gd_reduced_train), r2_score(y_test, pred_gd_reduced_test)]}, index=['Train', 'Test']).to_string())
test_history_reduced = np.mean((A_test_reduced @ wh_reduced.T - y_test.to_numpy()[:, None]) ** 2, axis=0)
plt.figure(figsize=(9, 5))
plt.plot(history_reduced, label='Train')
plt.plot(test_history_reduced, label='Test')
plt.yscale('log')
plt.xlabel('Итерация')
plt.ylabel('MSE (логарифмическая шкала)')
plt.title('Сходимость batch GD без permeability')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Вывод:** Реализован batch градиентный спуск с нулевыми начальными коэффициентами. Для полного набора шаг 0,1 и 3000 итераций выбраны по train; отличие коэффициентов от МНК около 1,3·10⁻¹². RMSE/R²: train 197,66/0,9613, test 208,98/0,9484.

Для сокращённого набора сравнили те же шаги на train, выбрали 0,1 при 3000 итерациях по близости к МНК. Максимальное отличие коэффициентов — 9,24·10⁻¹³. RMSE/R²: train 230,36/0,9475, test 237,50/0,9333. Графики показывают MSE на train и test, включая начальное состояние; test не использовался для выбора шага.

В обоих экспериментах прогноз совпадает с МНК до точности вычислений; улучшения не наблюдается. МНК минимизирует сумму квадратов ошибок на train среди линейных моделей с теми же признаками; другой набор весов может дать лучший или худший результат на test. Один набор и ранее проведённый EDA всей таблицы ограничивают независимость и обобщаемость оценки.

## L2-регуляризация

In [ ]:
def batch_gradient_descent_l2(X, y, learning_rate, iterations, alpha):
    weights = np.zeros(X.shape[1])
    for step in range(iterations):
        errors = X @ weights - y
        penalty = weights.copy()
        penalty[0] = 0
        gradient = (2 / len(y)) * X.T @ errors + 2 * alpha * penalty
        weights = weights - learning_rate * gradient
        if not np.all(np.isfinite(weights)) or np.max(np.abs(weights)) > 1e50:
            raise ValueError('Градиентный спуск разошёлся')
    return weights

alphas = [0.0, 0.001, 0.01, 0.1, 1.0]
cv_l2 = KFold(n_splits=5, shuffle=True, random_state=42)

def compare_l2_cv(X_train_fold, y_train_fold):
    rows = []
    for alpha in alphas:
        fold_rmse = []
        for train_idx, val_idx in cv_l2.split(X_train_fold):
            fold_train, fold_val = X_train_fold.iloc[train_idx], X_train_fold.iloc[val_idx]
            y_fold_train = y_train_fold.iloc[train_idx].to_numpy()
            y_fold_val = y_train_fold.iloc[val_idx].to_numpy()
            fold_imputer = SimpleImputer(strategy='median')
            fold_train = fold_imputer.fit_transform(fold_train)
            fold_val = fold_imputer.transform(fold_val)
            fold_scaler = StandardScaler()
            fold_train = fold_scaler.fit_transform(fold_train)
            fold_val = fold_scaler.transform(fold_val)
            fold_train = np.column_stack((np.ones(len(fold_train)), fold_train))
            fold_val = np.column_stack((np.ones(len(fold_val)), fold_val))
            fold_weights = batch_gradient_descent_l2(fold_train, y_fold_train, 0.1, 3000, alpha)
            fold_rmse.append(root_mean_squared_error(y_fold_val, fold_val @ fold_weights))
        rows.append({'alpha': alpha, 'RMSE CV среднее': np.mean(fold_rmse), 'RMSE CV std': np.std(fold_rmse)})
    return pd.DataFrame(rows)

cv_l2_results = compare_l2_cv(X_train, y_train)
alpha_selected = cv_l2_results.loc[cv_l2_results['RMSE CV среднее'].idxmin(), 'alpha']
print('Подбор alpha: 5-fold CV только на train; imputer и scaler обучаются внутри каждого фолда.')
print(cv_l2_results.to_string(index=False))
print('Выбранный alpha:', alpha_selected)

In [ ]:
weights_l2 = batch_gradient_descent_l2(X_gd_train, y_gd_train, 0.1, 3000, alpha_selected)
penalty_matrix = np.diag([0.0] + [1.0] * (X_gd_train.shape[1] - 1))
weights_l2_reference = np.linalg.solve(X_gd_train.T @ X_gd_train / len(y_gd_train) + alpha_selected * penalty_matrix, X_gd_train.T @ y_gd_train / len(y_gd_train))
assert np.allclose(weights_l2, weights_l2_reference, atol=1e-6, rtol=1e-8)
y_pred_l2_train = X_gd_train @ weights_l2
y_pred_l2_test = X_gd_test @ weights_l2
metrics_l2 = pd.DataFrame([
    ['Без регуляризации', 'Train', root_mean_squared_error(y_train, y_pred_gd_train), r2_score(y_train, y_pred_gd_train)],
    ['L2', 'Train', root_mean_squared_error(y_train, y_pred_l2_train), r2_score(y_train, y_pred_l2_train)],
    ['Без регуляризации', 'Test', root_mean_squared_error(y_test, y_pred_gd_test), r2_score(y_test, y_pred_gd_test)],
    ['L2', 'Test', root_mean_squared_error(y_test, y_pred_l2_test), r2_score(y_test, y_pred_l2_test)]
], columns=['Модель', 'Выборка', 'RMSE', 'R²'])
coef_l2 = pd.DataFrame({'Без регуляризации': weights_gd, 'L2': weights_l2, 'Разница': weights_l2 - weights_gd}, index=X_train_with_intercept.columns)
print('Сравнение после выбора alpha по CV на train:')
print(metrics_l2.to_string(index=False))
print('Коэффициенты стандартизированных признаков и свободный член:')
print(coef_l2.to_string())
print('Максимальное отличие от аналитического L2:', np.max(np.abs(weights_l2 - weights_l2_reference)))

In [ ]:
cv_reduced_results = compare_l2_cv(X_train_reduced, y_train)
positive_cv = cv_reduced_results[cv_reduced_results['alpha'] > 0]
alpha_reduced = positive_cv.loc[positive_cv['RMSE CV среднее'].idxmin(), 'alpha']
w_l2_reduced = batch_gradient_descent_l2(A_train_reduced, y_train.to_numpy(), 0.1, 3000, alpha_reduced)
penalty_reduced = np.diag([0.0] + [1.0] * (A_train_reduced.shape[1] - 1))
w_l2_reduced_ref = np.linalg.solve(A_train_reduced.T @ A_train_reduced / len(y_train) + alpha_reduced * penalty_reduced, A_train_reduced.T @ y_train.to_numpy() / len(y_train))
assert np.allclose(w_l2_reduced, w_l2_reduced_ref, atol=1e-6, rtol=1e-8)
print('CV без permeability:')
print(cv_reduced_results.to_string(index=False))
print('Лучший положительный alpha:', alpha_reduced)
models = [('Полный без L2', X_gd_train, X_gd_test, weights_gd), ('Полный L2', X_gd_train, X_gd_test, weights_l2), ('Без permeability', A_train_reduced, A_test_reduced, w_gd_reduced), ('Без permeability с L2', A_train_reduced, A_test_reduced, w_l2_reduced)]
comparison_rows = []
for name, train_matrix, test_matrix, weights in models:
    for sample, matrix, target in [('Train', train_matrix, y_train), ('Test', test_matrix, y_test)]:
        pred = matrix @ weights
        comparison_rows.append({'Модель': name, 'Выборка': sample, 'RMSE': root_mean_squared_error(target, pred), 'R²': r2_score(target, pred)})
print(pd.DataFrame(comparison_rows).to_string(index=False))
print('Коэффициенты сокращённой модели:')
print(pd.DataFrame({'Без L2': w_gd_reduced, 'L2': w_l2_reduced}, index=X_train_reduced_intercept.columns).round(4).to_string())
print('Максимальное отличие от аналитического L2:', np.max(np.abs(w_l2_reduced - w_l2_reduced_ref)))

**Вывод:** Для полного и сокращённого наборов применён ручной batch GD с L2-штрафом: MSE + α·Σwⱼ², свободный член не штрафуется. Медианное заполнение и масштабирование обучались внутри каждого из пяти фолдов train. Для полного набора CV выбрала α=0,001: RMSE 205,201 против 205,229 без штрафа. Для сокращённого лучший положительный α=0,01: 238,289 против 238,368 без штрафа. Различия малы по сравнению с разбросом между фолдами; выбор не доказывает преимущество регуляризации.

На train/test RMSE полного набора без штрафа — 197,658/208,983, с L2 — 197,661/209,082. Без permeability — 230,360/237,498, с L2 — 230,674/238,463. Соответствующие R² приведены в таблице. После выбора по CV test используется только для оценки; для этого разбиения L2 не улучшила тестовый RMSE в обеих ветках. Аналитическая проверка L2 совпадает с градиентным решением с точностью вычислений.

Для полного набора коэффициент porosity изменился с 685,291 до 683,324, permeability — с 194,715 до 195,639. В сокращённом наборе porosity изменилась с 858,077 до 842,743. L2 не обязана уменьшать каждый отдельный коэффициент. Полный набор показывает меньший RMSE на CV в этом сравнении, но сохраняет выявленную коррелирующую пару; сокращённый разрывает её ценой более высокой ошибки. Одна разбивка и ограниченная сетка α не дают универсального ответа. Кроме того, исходный EDA проводился по всей таблице до split, поэтому test не полностью независим для окончательного выбора; нужна новая независимая выборка для более надёжной оценки.